# AMD/NVIDIA SEC 历史索引边界机械修复

NVIDIA 官方目录声明边界比实际最后申报日晚一天；复用原件，只请求剩余 AMD 历史分卷并保留原失败记录。

In [1]:
import json,hashlib,subprocess,tarfile,tempfile,shlex,datetime
from pathlib import Path
from src.data.nas_market_data import _settings,nas_api_call
root=Path.cwd();card=root/'research/stage1_amd_nvda_sec_guidance_original_probe_20261023';raw=root/'data/snapshots/stage1_amd_nvda_sec_guidance_original_probe_20261023';repair=json.loads((card/'older_submissions_repair_plan.json').read_text());sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert repair['frozen'] and sha(card/'older_submissions_plan.json')==repair['original_plan_sha256'] and sha(card/'older_submissions_result.json')==repair['failed_result_sha256'] and sha(raw/'nvda_older_submissions.json')==repair['saved_NVDA_raw_sha256'] and not (card/'older_repair_state.json').exists()
state={'status':'started','cumulative_GET_attempts':1,'repair_GET_attempts':0,'model_fits':0,'native_accounts':0,'market_outcome_reads':0};(card/'older_repair_state.json').write_text(json.dumps(state,indent=2)+'\n')
spec=repair['remaining_exact_request'];body=raw/'amd_older_submissions.json';headers=raw/'amd_older_submissions.headers';assert not body.exists();state['cumulative_GET_attempts']=2;state['repair_GET_attempts']=1;(card/'older_repair_state.json').write_text(json.dumps(state,indent=2)+'\n');started=datetime.datetime.now(datetime.timezone.utc).isoformat()
cmd=['curl','--http1.1','--noproxy','*','--connect-timeout','10','--max-time','60','--max-filesize','8000000','--fail-with-body','-sS','-L','-D',str(headers),'-o',str(body),'-A',repair['request']['user_agent'],'-w','%{http_code} %{url_effective}',spec['url']]
try:
 run=subprocess.run(cmd,capture_output=True,text=True,timeout=75);parts=run.stdout.rsplit(' ',1);status=int(parts[0]) if len(parts)==2 and parts[0].isdigit() else None
 receipt={'issuer':'AMD','url':spec['url'],'requested_at_utc':started,'received_at_utc':datetime.datetime.now(datetime.timezone.utc).isoformat(),'curl_exit_code':run.returncode,'http_status':status,'final_url':parts[-1] if parts else None,'stderr':run.stderr[:250]}
except Exception as exc:receipt={'issuer':'AMD','url':spec['url'],'requested_at_utc':started,'received_at_utc':datetime.datetime.now(datetime.timezone.utc).isoformat(),'exception':str(exc)[:250]}
receipt.update(raw_sha256=sha(body) if body.exists() else None,raw_bytes=body.stat().st_size if body.exists() else 0,headers_sha256=sha(headers) if headers.exists() else None);(card/'older_repair_receipt.json').write_text(json.dumps(receipt,indent=2)+'\n')
checks=[];failure=None
if receipt.get('curl_exit_code')!=0 or receipt.get('http_status')!=200 or not body.exists():failure='AMD_transport'
else:
 try:
  original=json.loads((card/'older_submissions_plan.json').read_text())
  for spec in original['exact_requests']:
   name=spec['issuer'].lower();p=raw/f'{name}_older_submissions.json';d=json.loads(p.read_text());main=json.loads((raw/f'{name}_main_submissions.json').read_text())['filings']['recent'];n=len(d['accessionNumber']);keys=('accessionNumber','filingDate','form','acceptanceDateTime','primaryDocument');assert n==spec['expected_catalog_count'] and all(len(d[k])==n for k in keys) and len(set(d['accessionNumber']))==n and not set(d['accessionNumber'])&set(main['accessionNumber'])
   dates=[datetime.date.fromisoformat(x) for x in d['filingDate']];lo,hi=min(dates),max(dates);from_date=datetime.date.fromisoformat(spec['expected_filing_from']);to_date=datetime.date.fromisoformat(spec['expected_filing_to']);assert from_date<=lo<=hi<=to_date
   filings=[(x,y) for source in (d,main) for x,y in zip(source['filingDate'],source['form']) if '2011'<=x[:4]<='2026' and y=='8-K'];years={x[:4] for x,y in filings};assert all(str(y) in years for y in range(2011,2027))
   checks.append({'issuer':spec['issuer'],'raw_sha256':sha(p),'older_rows':n,'catalog_from':str(from_date),'catalog_to':str(to_date),'actual_first':str(lo),'actual_last':str(hi),'start_edge_gap_days':(lo-from_date).days,'end_edge_gap_days':(to_date-hi).days,'combined_2011_2026_8K_count':len(filings),'8K_years':len(years)})
 except Exception as exc:failure='schema_'+type(exc).__name__+'_'+str(exc)[:100]
(card/'older_repair_checks.json').write_text(json.dumps(checks,indent=2)+'\n')
files=[{'name':p.name,'sha256':sha(p),'bytes':p.stat().st_size} for p in sorted(raw.iterdir()) if p.name in ('nvda_older_submissions.json','nvda_older_submissions.headers','amd_older_submissions.json','amd_older_submissions.headers')];manifest={'plan_sha256':sha(card/'older_submissions_repair_plan.json'),'files':files,'receipt_sha256':sha(card/'older_repair_receipt.json')};(card/'older_repair_manifest.json').write_text(json.dumps(manifest,indent=2)+'\n')
settings=_settings(root)
with tempfile.TemporaryDirectory(prefix='amd-nvda-older-repair-') as td:
 archive=Path(td)/'source.tar.gz'
 with tarfile.open(archive,'w:gz') as t:
  for x in files:t.add(raw/x['name'],arcname='raw/'+x['name'])
  for name in ('older_submissions_repair_plan.json','older_repair_receipt.json','older_repair_manifest.json'):t.add(card/name,arcname=name)
 digest=sha(archive);a=nas_api_call(origin=settings['origin'],token=settings['token'],method='PUT',path='/api/research/artifacts/'+digest,upload=archive,executor_id='soxl-minute-research')['artifact'];assert a['sha256']==digest and a['size_bytes']==archive.stat().st_size
 remote=settings['artifact_root'].rstrip('/')+'/'+digest[:2]+'/'+digest[2:4]+'/'+digest
 with tempfile.TemporaryDirectory(prefix='amd-nvda-older-repair-restore-') as rd:
  restored=Path(rd)/'restore.tar.gz'
  with restored.open('wb') as out:run=subprocess.run(['ssh','-p',str(settings['ssh']['port']),'-o','BatchMode=yes',settings['ssh']['host'],'cat '+shlex.quote(remote)],stdout=out,stderr=subprocess.PIPE,timeout=120)
  assert run.returncode==0 and sha(restored)==digest
  with tarfile.open(restored,'r:gz') as t:
   for x in files:
    m=t.extractfile('raw/'+x['name']);assert m and hashlib.sha256(m.read()).hexdigest()==x['sha256']
result={'status':'AMD_NVDA_SEC_8K_metadata_2011_2026_coverage_passed_catalog_edge_repair' if not failure and len(checks)==2 else 'bounded_AMD_NVDA_history_repair_partial_or_failed','failure':failure,'plan_sha256':sha(card/'older_submissions_repair_plan.json'),'cumulative_GET_attempts':2,'repair_GET_attempts':1,'checks':checks,'initial_partial_NAS_archive_sha256':json.loads((card/'older_submissions_result.json').read_text())['NAS_archive_sha256'],'repair_NAS_archive_sha256':digest,'NAS_remote_and_scoped_restore_verified':True,'original_quarterly_guidance_coverage_qualified':False,'market_outcome_reads':0,'model_fits':0,'native_accounts':0};(card/'older_repair_result.json').write_text(json.dumps(result,indent=2)+'\n');state.update(status='completed' if not failure else 'partial_or_failed',NAS_archive_sha256=digest);(card/'older_repair_state.json').write_text(json.dumps(state,indent=2)+'\n');print({'status':result['status'],'failure':failure,'GETs':state['cumulative_GET_attempts'],'checks':checks,'NAS':digest})


{'status': 'AMD_NVDA_SEC_8K_metadata_2011_2026_coverage_passed_catalog_edge_repair', 'failure': None, 'GETs': 2, 'checks': [{'issuer': 'NVDA', 'raw_sha256': '4a3b5cc6b698ff2e49cd4ef95539964b0690bccc5882ac3db8ffd2c4b7fce99c', 'older_rows': 1489, 'catalog_from': '1998-03-06', 'catalog_to': '2020-09-15', 'actual_first': '1998-03-06', 'actual_last': '2020-09-14', 'start_edge_gap_days': 0, 'end_edge_gap_days': 1, 'combined_2011_2026_8K_count': 152, '8K_years': 16}, {'issuer': 'AMD', 'raw_sha256': '8d784eaa4df0f1af2b3a2cc2964ad59d2702fb7f2c066737b6c322cbc3725530', 'older_rows': 2001, 'catalog_from': '2003-08-26', 'catalog_to': '2017-12-16', 'actual_first': '2003-08-26', 'actual_last': '2017-12-14', 'start_edge_gap_days': 0, 'end_edge_gap_days': 2, 'combined_2011_2026_8K_count': 213, '8K_years': 16}], 'NAS': '62676e05e11c72041033a4d1ad8af3da78248028a303c2b6d7ca8a1d1ecf8998'}


## 中文结论

只修官方目录范围包含关系；复用 NVIDIA 原件，仅请求 AMD 分卷后两家 2011–2026 每年 8-K 元数据覆盖通过，NAS 恢复通过。 完整决定见 `conclusion.md`。